# Build h3_density
Runs `build_h3_density()` standalone and outputs `h3_density.csv`.
Use this to inspect/test the aggregation without running the full ETL.

Run from the `server/` directory kernel (same venv as ETL).

In [1]:
import sys
from pathlib import Path

# Ensure server/ is on sys.path so db.etl.* imports resolve
SERVER_ROOT = Path.cwd()
while SERVER_ROOT.name != 'server' and SERVER_ROOT != SERVER_ROOT.parent:
    SERVER_ROOT = SERVER_ROOT.parent
if str(SERVER_ROOT) not in sys.path:
    sys.path.insert(0, str(SERVER_ROOT))

print('SERVER_ROOT:', SERVER_ROOT)

SERVER_ROOT: c:\Users\kylec\OneDrive\Desktop\react_project\london-explorer\server


In [3]:
from db.etl.load_places import load_places
from db.etl.build_h3_density import build_h3_density

df = load_places()

  13,092 rows loaded (12,320 open, 772 temporarily closed)


In [4]:
density_df = build_h3_density(
    df[[
        'h3_res10', 'lat', 'lon', 'cuisineType', 'cost', 'venueType',
        'normal_1', 'tier', 'tier_d', 'tier_independent',
    ]].rename(columns={
        'h3_res10':    'h3_r10',
        'cuisineType': 'cuisine_type',
        'venueType':   'venue_type',
    }).copy()
)

print(f'Total rows: {len(density_df):,}')
density_df.head()

  98,400 / 98,400 combos processed … done
Total rows: 1,590,022


,tile,resolution,cuisine_type,cost,venue_type,score_basis,score_tier,count,agg_lat,agg_lon
0,87195da69ffffff,7,African,10+,Dine-In,0,0,5,51.548345,-0.077656
1,87195da68ffffff,7,African,10+,Dine-In,0,0,12,51.563391,-0.111119
2,87194ad33ffffff,7,African,10+,Dine-In,0,0,9,51.489893,-0.089349
3,87194ad15ffffff,7,African,10+,Dine-In,0,0,5,51.479195,-0.096945
4,87195da6bffffff,7,African,10+,Dine-In,0,0,11,51.547757,-0.113703


In [5]:
# Quick sanity checks
print('score_basis values:', sorted(density_df['score_basis'].unique()))
print('score_tier values: ', sorted(density_df['score_tier'].unique()))
print('resolution values: ', sorted(density_df['resolution'].unique()))
print('cuisine_type sample:', sorted(density_df['cuisine_type'].unique())[:6])
print('Duplicate PK rows:  ', density_df.duplicated(
    subset=['tile','resolution','cuisine_type','cost','venue_type','score_basis','score_tier']
).sum())

score_basis values: [np.int64(0), np.int64(1), np.int64(2)]
score_tier values:  [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
resolution values:  [np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11)]
cuisine_type sample: ['African', 'American', 'Asian', 'Australian', 'Bakery & Pastry', 'Bar & Pub']
Duplicate PK rows:   0


In [6]:
density_df['cuisine_type'].unique()

<ArrowStringArray>
[            'African',            'American',               'Asian',
          'Australian',     'Bakery & Pastry',           'Bar & Pub',
              'Bistro',             'British',  'Brunch & Breakfast',
              'Buffet',             'Burgers',       'Cafe & Coffee',
             'Chinese', 'Dessert & Ice Cream',    'Eastern European',
            'European',   'Family Restaurant',           'Fast Food',
         'Fine Dining',              'French',              'German',
               'Halal',             'Italian',            'Japanese',
          'Kebab Shop',              'Korean',      'Latin American',
       'Mediterranean',      'Middle Eastern',   'Northern European',
               'Pizza',     'Sandwich & Deli',             'Seafood',
         'South Asian',     'Southeast Asian',   'Southern European',
    'Steakhouse & BBQ',               'Tapas',  'Vegetarian & Vegan',
            '__null__',             '__all__']
Length: 41, dtype: str

In [7]:
OUT_PATH = SERVER_ROOT / 'out' / 'h3_density.csv'
density_df.to_csv(OUT_PATH, index=False)
print(f'Saved to {OUT_PATH}')

Saved to c:\Users\kylec\OneDrive\Desktop\react_project\london-explorer\server\out\h3_density.csv
